
# Automated API Testing, Validation & Documentation Engine

## AI Agent with vLLM + PydanticAI + MCP

This notebook builds a complete AI system for:
- Automated API testing
- Response validation
- cURL generation
- Documentation generation
- Handling failure scenarios


## Step 1: Install Dependencies

In [ ]:
!pip install -q 'pydantic-ai-slim[openai]' requests pandas

## Step 2: Configure vLLM Connection

In [ ]:
import os
BASE_URL = 'http://localhost:8000/v1'
os.environ['OPENAI_API_KEY'] = 'abc-123'

## Step 3: Initialize Model

In [ ]:
from pydantic_ai.models.openai import OpenAIChatModel
from pydantic_ai.providers.openai import OpenAIProvider
provider = OpenAIProvider(base_url=BASE_URL, api_key='abc-123')
model = OpenAIChatModel('Qwen3-4B', provider=provider)

## Step 4: Create Agent

In [ ]:
from pydantic_ai import Agent
agent = Agent(model=model)

## Step 5: Async Runner

In [ ]:
import asyncio
async def run_async(prompt):
    result = await agent.run(prompt)
    return result.output

## Step 6: API Execution Tool

In [ ]:
import requests
from pydantic_ai import Tool

@Tool
def execute_api(url: str, method: str, payload: dict):
    try:
        if method == 'GET':
            res = requests.get(url)
        else:
            res = requests.post(url, json=payload)
        return {'status': res.status_code, 'response': res.text}
    except Exception as e:
        return {'status': 'ERROR', 'response': str(e)}

## Step 7: Validation Tool

In [ ]:
@Tool
def validate_response(actual_status: int, expected_status: int):
    return {'result': 'PASS' if actual_status == expected_status else 'FAIL'}

## Step 8: cURL Generator

In [ ]:
@Tool
def generate_curl(url: str, method: str, payload: dict):
    return f'curl -X {method} {url} -d {payload}'

## Step 9: Documentation Generator

In [ ]:
@Tool
def generate_doc(url: str, payload: dict, status: int):
    return {'endpoint': url, 'request': payload, 'expected_status': status}

## Step 10: Build AI Agent

In [ ]:
agent = Agent(model=model, tools=[execute_api, validate_response, generate_curl, generate_doc])

## Step 11: Successful Test Case

In [ ]:
result = await run_async('Test API https://jsonplaceholder.typicode.com/posts GET expected 200')
print(result)

## Step 12: Failed Test Case - Wrong Endpoint

In [ ]:
result = await run_async('Test API https://wrongurl.typicode.com/posts GET expected 200')
print(result)

## Step 13: Failed Test Case - Wrong Expected Status

In [ ]:
result = await run_async('Test API https://jsonplaceholder.typicode.com/posts GET expected 404')
print(result)

## Step 14: Failed Test Case - Invalid Payload

In [ ]:
result = await run_async('Test API https://jsonplaceholder.typicode.com/posts POST invalid payload expected 201')
print(result)

## Step 15: Reporting Sample

In [ ]:
report = {
    'total_tests': 3,
    'passed': 1,
    'failed': 2
}
print(report)